### Libraries

In [112]:
import matplotlib.pyplot as plt
import numpy as np
import random
import os
from typing import Tuple
from sklearn.preprocessing import PolynomialFeatures
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn import linear_model

In [113]:
seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

# 2. Funkcje kosztu i regresja liniowa

## Podział zbioru danych

Aby rzetelnie ocenić model i uniknąć **przeuczenia** (overfittingu), nie możemy testować modelu na tych samych danych, na których go trenowaliśmy. Standardowo zbiór danych dzieli się na dwie lub trzy części.

### 1. Zbiór Treningowy (Training Set)
Zazwyczaj **70%-80%** całości danych. Służy do trenowania modelu. Algorytm szuka w nim wzorców i dopasowuje swoje parametry (np. wagi w regresji). Analogią może być  podręcznik, z którego student przygotowuje się do egzaminu.

### 2. Zbiór Walidacyjny (Validation Set)
Zazwyczaj **10%-15%** (opcjonalny). Służy do szukania **hiperparametrów** modelu oraz monitorowania, czy model nie zaczyna się "uczyć na pamięć". Zazwyczaj jest częścią zbioru treningowego i jest do niego później zwracany. To próbny arkusz egzaminacyjny, który student rozwiązuje w domu, by sprawdzić postępy i poprawić technikę.

### 3. Zbiór Testowy (Test Set)
Zazwyczaj **10%-20%** całości danych. To ostateczny sprawdzian dla modelu, dane te są całkowicie "niewidoczne" dla niego podczas całego procesu nauki i dopasowywania. Wyliczamy na nim końcowe metryki ($R^2$, $RMSE$ itp.) i je oceniamy. Analogicznie to oficjalny egzamin - student widzi pytania pierwszy raz w życiu.

### Wskazówki:
1.  **Mieszanie (Shuffle):** Zawsze mieszaj dane przed podziałem, aby uniknąć sytuacji, w której np. w zbiorze testowym znajdą się dane tylko z jednego miesiąca lub jednej kategorii.
2.  **Random State:** W kodzie (np. `train_test_split`) zawsze ustawiaj parametr `random_state`, aby Twój podział był **reprodukowalny** (za każdym razem taki sam).
3.  **Wyciek danych:** Nigdy nie pozwól, aby informacje ze zbioru testowego "wyciekły" do treningowego.


In [ ]:
class Dataset:
    def __init__(self, data, target):
        self.data = data
        self.target = target

def embed_poly(X, poly_degree=3):
    poly = PolynomialFeatures(degree=poly_degree, include_bias=False)
    return poly.fit_transform(X)

def create_regression_dataset(func, sample_size=100, embed_func=None, embed_kwargs=None):
    dataset_X = np.random.uniform(-2.5, 2.5, size=(sample_size, 1))
    dataset_Y_clean = func(dataset_X)
    dataset_Y = dataset_Y_clean + np.random.normal(0, 0.5, size=dataset_Y_clean.shape)
    dataset_Y = dataset_Y.squeeze()
    if embed_func is not None:
        dataset_X = embed_func(dataset_X, **embed_kwargs)
    return Dataset(dataset_X, dataset_Y)

def plot_regression_dataset(dataset, name):
  plt.plot([dataset.data.min(), dataset.data.max()], [0, 0], "k--")
  plt.plot([0, 0], [dataset.target.min(), dataset.target.max()], "k--")
  plt.title(name)
  plt.scatter(dataset.data, dataset.target)
  plt.show()

def cubic_func(X):
    return X ** 3 - 2 * X

dataset = create_regression_dataset(cubic_func, 500, embed_poly, {"poly_degree": 5})
X_linspace = np.linspace(-4.0, 4.0)
plt.plot(X_linspace, cubic_func(X_linspace))
plt.title("Generated function")
plt.show()

Przygotować funkcję do podziału danych (nie używać gotowych funkcji z dostępnych bibliotek).

Podziel zbiór danych:
* 70% na zbiór treningowy.
* 10% na zbiór walidacyjny.
* 20% na zbiór testowy.

Funkcja ma zwracać trzy obiekty: train_set, valid_set, test_set.

In [ ]:
def split_data(data_x: np.ndarray, data_y: np.ndarray):
    order = ...
    np.random.shuffle(order)
    data_x = ...
    data_y = ...

    valid_start = ...
    test_start = ...

    train_set = ...
    valid_set = ...
    test_set = ...

    return train_set, valid_set, test_set

train_set, valid_set, test_set = split_data(dataset.data, dataset.target)


## Funkcje kosztu

Funkcja kosztu to funkcja oceny wyniku zwróconego przez Nasz model.

Kluczowe metryki do oceny problemu regresji  to:

### 1. R-kwadrat ($R^2$ - Coefficient of Determination)
* Mierzy proporcję wariancji zmiennej zależnej, która jest wyjaśniana przez model.
* **Zakres:** Zazwyczaj od 0 do 1. Im bliżej 1, tym lepsze dopasowanie modelu do danych.
* **Wzór:** $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$
* **Ograniczenie:** Może sztucznie wzrastać przy dodawaniu nowych zmiennych, nawet jeśli nie wnoszą one realnej wartości informacyjnej.

### 2. Błąd średniokwadratowy (MSE - Mean Squared Error)
* Średnia z kwadratów różnic między wartościami przewidywanymi ($\hat{y}$) a rzeczywistymi ($y$).
* **Wzór:** $MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$
* **Właściwość:** Bardzo wrażliwy na **wartości odstające** (outliery), ponieważ błędy są podnoszone do kwadratu.

### 3. Pierwiastek błędu średniokwadratowego (RMSE - Root Mean Squared Error)
* Pierwiastek kwadratowy z MSE.
* **Zaleta:** Wynik jest wyrażony w tych samych jednostkach co zmienna docelowa (np. USD, kg, m), co ułatwia interpretację.
* **Wzór:** $RMSE = \sqrt{MSE}$

### 4. Średni błąd bezwzględny (MAE - Mean Absolute Error)
* Średnia z bezwzględnych różnic między wartościami przewidywanymi a rzeczywistymi.
* **Wzór:** $MAE = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$
* **Właściwość:** Bardziej "odporny" na wartości odstające niż MSE/RMSE.

In [ ]:
def mean_squared_error(y_true, y_pred):
    ...
    return mse

def root_mean_squared_error(y_true, y_pred):
    ...
    return rmse

def mean_absolute_error(y_true, y_pred):
    ...
    return mae

def r2_score(y_true, y_pred):
    ...
    return r2

In [ ]:
y_t = np.array([1, 2, 3])
y_p = np.array([1, 2, 5])

assert np.allclose(mean_squared_error(y_t, y_p), 4/3)
assert np.allclose(mean_absolute_error(y_t, y_p), 2/3)
assert np.allclose(root_mean_squared_error(y_t, y_p), np.sqrt(4/3))
assert np.allclose(r2_score(y_t, y_p), -1.0)

## Regresja liniowa

Zaimplementować od podstaw klasę regresji liniowej, która naśladuje interfejs znany z biblioteki `scikit-learn`.

1. Metoda `fit(X, y)` - znalezienie optymalnego wektora wag $\mathbf{w}$, który najlepiej dopasuje model do danych treningowych. Należy obliczyć wektor wag rozwiązujących problem regresji dla podanej macierzy $\mathbf{X}$ o wymiarach $[N, D]$ oraz dla podanego wektora etykiet $\mathbf{y}$ o wymiarze $[N]$.Należy skorzystać z poniższego jawnego wzoru:
  $$\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$$
Do odwracania macierzy w NumPy można użyć `np.linalg.inv()`.

2. Metoda `predict(X)` - służy do generowania prognoz dla nowych danych wejściowych. Dla podanej macierzy numpy $\mathbf{X}$ o wymiarach $[N, D]$ należy wyznaczyć wektor predykcji. Należy obliczyć iloczyn macierzowy danych wejściowych i wyznaczonych wcześniej wag. Metoda ma zwracać wektor predykcji o wymiarze $[N]$.

In [ ]:
class LinearRegression:
    def __init__(self):
        self.weight = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        ...

    def predict(self, X: np.ndarray) -> np.ndarray:
        ...


Dla wygnerowanego datasetu:
1. Nauczyć model regresji i przewidzieć wartości y
2. Wypisac błąd MAE (użyć wcześcniej zaimplementowanej funkcji)
3. Wyrysować linię regresji wraz z punktami.

In [ ]:
np.random.seed(42)
X = np.random.rand(50, 1) * 10
y = 2.5 * X + np.random.randn(50, 1) * 2

In [ ]:
model = ...
...
y_pred = ...

mse  = ...
plt.figure(figsize=(8,6))
plt.scatter(X, y, color='blue', label='Data Points')
plt.plot(X, y_pred, color='red', linewidth=2, label='Regression Line')
plt.title('Linear Regression on Random Dataset')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.grid(True)
plt.show()

## Przykład z zaimportowanym zbiorem danych

Najpierw załadujemy przykład danych z pomiarami glukozy, na tym etapie nie będziemy jeszcze analizować danych i ich dostosowywać, skupimy się na tym w innym ćwiczeniu.

In [ ]:
from sklearn.datasets import load_diabetes
import pandas as pd

data = load_diabetes()

df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(f"Liczba rekordów: {len(df)}")
df.head()

Następnie dzielimy zbiór na treningowy, testowy oraz walidacyjny, można użyć przygotowanej funkcji lub tej z biblioteki sklern, nauczyć model i sprawdzić metryki an zbiorze treningowym i testowym.

In [ ]:
X = data.data
y = data.target

X_train, X_test, y_train, y_test = ...

model = linear_model.LinearRegression()
model.fit(...
...

## K-krotna walidacja krzyżowa

Napisać funkcję k-krotnej walidacji krzyżowej i przetestować na powyższym zbiorze danych.

In [ ]:
def kfold_cv(X, y, k=5):
    ...
    return np.mean(errors)


In [ ]:
kfold_cv(X, y, k=5)